# 00_01 Setup: get the course running on your computer

*Our own page: not in the book.*

You only do this once. Most of this page is reading, because you can't run a notebook until setup is done. Then the last cell checks that it all works.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **getting ready**: nothing to build yet
>
> Project files this connects to:
>
> - `pyproject.toml`
> - `.vscode/extensions.json`
>
> **This notebook:** No notebook will run until your computer has the project, Python and the libraries. This page gets you there, then checks it worked.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a terminal is, and how to open one
- How to install VS Code and uv, and get the project onto your computer
- How to open a notebook and pick the right kernel
- How to check that your setup works

## 2. The idea

### ❓ What is a terminal?

A **terminal** is a window where you type commands instead of clicking buttons. You type a command, press Enter, and the computer answers in text.

In VS Code it's the menu **Terminal → New Terminal**. On a Mac without VS Code, open the app called **Terminal**.

### ❓ Which programs do I need?

Three, all free:

| Program | What it is | Get it from |
|---|---|---|
| **VS Code** | the editor: where you read code and run notebooks | <https://code.visualstudio.com/> |
| **git** | the tool that copies the project and keeps its history | <https://git-scm.com/> (Macs often have it already) |
| **uv** | the tool that installs Python and every library the project needs | <https://docs.astral.sh/uv/> |

Not sure if you already have one? Type its name and `--version` in a terminal, like `git --version`, and see if it answers.

## 3. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How do I get the project onto my computer?

Copy it with git. In a terminal:

```
git clone https://github.com/IcodeNet/homer-ai
cd homer-ai
```

`git clone` gives you your own copy of the project folder, and `cd` (**change directory**) moves the terminal into it.

### ❓ How do I install everything the project needs?

Inside the project folder, run:

```
uv sync
```

uv reads `pyproject.toml`, the project's shopping list, and installs Python, PyTorch, tiktoken and the notebook tools into a folder called `.venv` (the project's own Python). The clone itself was small. This is the big download: about 900 MB, mostly PyTorch, and only the first time. `.venv` is never committed to git. You do this once, and again only if the list changes.

### ❓ How do I open the project in VS Code?

In VS Code, go to **File → Open Folder…** and pick the `homer-ai` folder. VS Code will then suggest the extensions this project uses (Python, Pylance, Ruff, Jupyter), because they're listed in `.vscode/extensions.json`. Say yes. An **extension** is an add-on that teaches VS Code a new trick, like running notebooks.

### ❓ VS Code asks me to "Select Kernel". Which one?

The **kernel** is the Python program that runs your notebook's cells in the background. Click **Select Kernel** (top right of the notebook) and choose the Python inside `.venv`. That's the one with PyTorch and our `homer_ai` code installed. VS Code remembers your choice for each notebook, so you only do it once per notebook.

### ❓ My notebook opens as a wall of text. What happened?

A notebook is stored as **JSON**, a plain-text format for data, and VS Code has opened it in the text editor. Right-click the file, choose **Open With…**, then **Jupyter Notebook**. Usually, double-clicking it in the file list opens it the right way anyway.

### ❓ Shift+Enter or Run All: which do I use?

- **Shift+Enter** runs only the cell you're in, then moves to the next one. Use it while you're learning, so you can predict each result first.
- **Run All** (the button at the top) runs every cell, top to bottom. Use it when you reopen a notebook. The kernel forgets everything when it restarts, and a cell can only use what the cells above it made.

## 4. Full example

### ❓ Is my setup working?

Pick the `.venv` kernel, then run the cell below. Every line should start with ✅. If one starts with ❌, it tells you what to fix.

You don't have to read this code. It only checks that things are installed, so just run it.

In [ ]:
import importlib
import sys
from pathlib import Path


def check(ok, good, bad):
    print(("✅ " + good) if ok else ("❌ " + bad))


# 1. Python itself
v = sys.version_info
version = f"Python {v.major}.{v.minor}"
check(v >= (3, 12), version, f"{version} is too old: run `uv sync`")

# 2. The libraries the course needs
for name in ["torch", "tiktoken"]:
    try:
        importlib.import_module(name)
        check(True, f"{name} is installed", "")
    except ImportError:
        check(False, "", f"{name} is missing: run `uv sync`, then pick .venv")

# 3. Our own code
try:
    import homer_ai.tokenizer  # noqa: F401

    check(True, "the homer_ai code imports", "")
except ImportError:
    check(False, "", "homer_ai doesn't import: pick the .venv kernel")

# 4. The Greek and English texts
project = Path.cwd()
while not (project / "pyproject.toml").exists() and project != project.parent:
    project = project.parent  # walk up from notebooks/... to the project folder

books = sorted((project / "data" / "homer").glob("*/*/book_*.txt"))
check(
    len(books) == 96,
    f"data/homer/ has all {len(books)} book files",
    f"data/homer/ has only {len(books)} of the 96 book files",
)
book_1 = project / "data" / "homer" / "odyssey" / "greek" / "book_01.txt"
check(
    book_1.exists(),
    "the training text is there (Book 1 of the Odyssey)",
    "data/homer/odyssey/greek/book_01.txt is missing",
)

# 5. The two trained models (yours if you made them, else the ready-made ones)
MODELS = [
    ("the small model", "homer_model.pt", "odyssey_book1.pt"),
    ("the GPT", "homer_gpt.pt", "odyssey_book1_gpt.pt"),
]
for what, own, ready_made in MODELS:
    if (project / "src" / "homer_ai" / own).exists():
        print(f"✅ {what}: your own is there ({own})")
    elif (project / "models" / ready_made).exists():
        print(f"✅ {what}: the ready-made one is there (models/{ready_made})")
    else:
        print(f"❌ models/{ready_made} is missing: `git status` shows what is gone")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

One line per check, and two for the models: the small one of chapter 1 and the GPT of chapter 4. If any line says ❌, fix that one thing and run the cell again. You shouldn't see a red `Failed to initialize NumPy` warning, because NumPy comes with the project. If you do, it's harmless (more in 00_02). And if you get an error message instead, the next notebook shows you how to read it.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Do I need a GPU?

No. A **GPU** is a graphics chip that does maths very fast, and big models need one. Ours is small enough for a laptop's normal processor, the CPU. Training the GPT on Book 1 takes about 3 minutes, and the small model about 8.

## 5. Recap

- A terminal is where you type commands. `git clone` copies the project and `uv sync` installs it.
- Open the folder in VS Code, accept the extensions and pick the `.venv` kernel.
- The check cell should show ✅ on every line.

**Next:** [00_02: Reading errors](00_02_reading_errors.ipynb).